# NB05 — MGnify vs SPIRE comparison (Exploratory)

**Status:** Exploratory analysis. Cross-dataset comparison of results.

**Goal:** Compare MGnify and SPIRE results side-by-side.

**Analyses:**
1. CV RMSE table (B0, M1, M2, M3 for each dataset).
2. Scatter plots of effect sizes across datasets.
3. Distribution of target variable (PF1_Cu — same for both datasets).
4. **H8:** Spearman ρ > 0.3 between SPIRE and MGnify genus-level PGLS β coefficients (if both PGLS analyses complete).

**Output:** `data/mgnify_vs_spire_comparison.csv`, figures.


In [1]:
print("NB05 executing — MGnify vs SPIRE comparison (exploratory).")

NB05 executing — MGnify vs SPIRE comparison (exploratory).


In [2]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, FIGW, ROW_H, PALETTE, grid_h
apply_style()

DATA_DIR = Path.cwd().parent / 'data'
FIG_DIR = Path.cwd().parent / 'figures'
FIG_DIR.mkdir(exist_ok=True)

In [3]:
# Load SPIRE and MGnify CV results
spire_cv = pd.read_csv(DATA_DIR / 'cv_results.csv')
spire_summary = spire_cv.groupby('model').agg({
    'rmse': ['mean', 'std'],
    'r2': ['mean', 'std'],
}).reset_index()
spire_summary.columns = ['model', 'rmse_mean', 'rmse_std', 'r2_mean', 'r2_std']
spire_summary['dataset'] = 'SPIRE'
spire_summary['target'] = 'PF1_Cu'

mgnify_cv = pd.read_csv(DATA_DIR / 'mgnify_mobility_prediction_results.csv')
mgnify_summary = mgnify_cv.groupby('model').agg({
    'rmse': ['mean', 'std'],
    'r2': ['mean', 'std'],
})
if len(mgnify_summary.columns) > 0:
    mgnify_summary = mgnify_summary.reset_index()
    mgnify_summary.columns = ['model', 'rmse_mean', 'rmse_std', 'r2_mean', 'r2_std']
    mgnify_summary['dataset'] = 'MGnify'
    mgnify_summary['target'] = 'PF1_Cu'  # MGnify uses same PF1_Cu target as SPIRE
    
    comparison = pd.concat([spire_summary, mgnify_summary], ignore_index=True)
else:
    print("MGnify CV results not available.")
    comparison = spire_summary.copy()

print("Cross-dataset CV RMSE comparison:")
print(comparison.to_string(index=False))

Cross-dataset CV RMSE comparison:
model  rmse_mean  rmse_std   r2_mean   r2_std dataset target
   B0   0.050099  0.021271 -0.171103 0.220705   SPIRE PF1_Cu
   B1   0.054715  0.018610 -0.540266 0.499944   SPIRE PF1_Cu
   B2   0.043901  0.021995 -0.029517 0.764373   SPIRE PF1_Cu
   M1   0.052682  0.019658 -0.341043 0.175496   SPIRE PF1_Cu
   M2   0.043901  0.021995 -0.029517 0.764373   SPIRE PF1_Cu
   M3   0.040016  0.018510  0.162105 0.442668   SPIRE PF1_Cu
   B0   0.036877  0.014692 -0.082795 0.079605  MGnify PF1_Cu
   M1   0.038528  0.013387 -0.231245 0.165184  MGnify PF1_Cu
   M2   0.037001  0.008953 -0.380569 0.790247  MGnify PF1_Cu
   M3   0.038652  0.008233 -0.523430 0.875887  MGnify PF1_Cu


In [4]:
# Save comparison table
comparison.to_csv(DATA_DIR / 'mgnify_vs_spire_comparison.csv', index=False)
print(f"Saved: {DATA_DIR / 'mgnify_vs_spire_comparison.csv'}")

Saved: /home/hmacgregor/BERIL-research-observatory/projects/metagenomic_environment_prediction/data/mgnify_vs_spire_comparison.csv


In [5]:
# Plot CV RMSE by dataset and model
fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

for ax, (dataset, data) in zip(axes, comparison.groupby('dataset')):
    models = data['model'].values
    rmse = data['rmse_mean'].values
    rmse_err = data['rmse_std'].values

    colours = [PALETTE[0] if m == 'B0' else PALETTE[1] for m in models]
    ax.bar(range(len(models)), rmse, yerr=rmse_err, color=colours, capsize=4, alpha=0.85,
           edgecolor='k', linewidth=0.5)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models)
    ax.set_xlabel('Model')
    ax.set_ylabel('Mean RMSE')
    ax.set_title(f'{dataset} — CV RMSE by model')
    ax.set_ylim(0, max(comparison['rmse_mean']) * 1.3)
    grid_h(ax)

save(fig, FIG_DIR / 'nb05_cv_rmse_comparison')

In [6]:
# Load feature matrices to compare target distributions
spire_df = pd.read_parquet(DATA_DIR / 'mag_feature_matrix.parquet')
mgnify_df = pd.read_csv(DATA_DIR / 'mgnify_mag_feature_matrix.csv')

fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

axes[0].hist(spire_df['PF1_Cu'].dropna(), bins=30, alpha=0.7, color=PALETTE[0], edgecolor='k')
axes[0].set_xlabel('PF1_Cu (metal mobility fraction)')
axes[0].set_ylabel('Frequency')
axes[0].set_title(f'SPIRE PF1_Cu (n={spire_df["PF1_Cu"].notna().sum()})')
grid_h(axes[0])

axes[1].hist(mgnify_df['PF1_Cu'].dropna(), bins=30, alpha=0.7, color=PALETTE[1], edgecolor='k')
axes[1].set_xlabel('PF1_Cu (CSU Cu mobility fraction)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'MGnify PF1_Cu (n={mgnify_df["PF1_Cu"].notna().sum()})')
grid_h(axes[1])

save(fig, FIG_DIR / 'nb05_target_distributions')

In [7]:
# H8: Spearman rho > 0.3 between SPIRE and MGnify genus-level PGLS beta coefficients
print("\n=== Hypothesis H8: Cross-dataset PGLS coefficient correlation ===")

spire_pgls = DATA_DIR / 'pgls_validation_results.csv'
mgnify_pgls = DATA_DIR / 'mgnify_pgls_validation.csv'

h8_supported = False
if spire_pgls.exists() and mgnify_pgls.exists():
    try:
        spire_pgls_df = pd.read_csv(spire_pgls)
        mgnify_pgls_df = pd.read_csv(mgnify_pgls)

        # Match on shared predictor names
        shared_predictors = set(spire_pgls_df['predictor']) & set(mgnify_pgls_df['predictor'])
        print(f"Shared predictors: {sorted(shared_predictors)}")

        if len(shared_predictors) >= 2:
            spire_sub = spire_pgls_df[spire_pgls_df['predictor'].isin(shared_predictors)].set_index('predictor')
            mgnify_sub = mgnify_pgls_df[mgnify_pgls_df['predictor'].isin(shared_predictors)].set_index('predictor')
            shared_idx = sorted(shared_predictors)
            spire_betas = spire_sub.loc[shared_idx, 'beta'].values
            mgnify_betas = mgnify_sub.loc[shared_idx, 'beta'].values
            rho, pval = spearmanr(spire_betas, mgnify_betas)
            print(f"Spearman rho (SPIRE vs MGnify beta): {rho:.3f} (p={pval:.4f})")
            h8_supported = rho > 0.3
            print(f"H8 (rho > 0.3): {'SUPPORTED' if h8_supported else 'NOT SUPPORTED'}")
        elif len(shared_predictors) == 1:
            pred = list(shared_predictors)[0]
            s_beta = spire_pgls_df.loc[spire_pgls_df['predictor'] == pred, 'beta'].values[0]
            m_beta = mgnify_pgls_df.loc[mgnify_pgls_df['predictor'] == pred, 'beta'].values[0]
            same_sign = (s_beta > 0) == (m_beta > 0)
            print(f"Only 1 shared predictor ({pred}): SPIRE beta={s_beta:.4f}, MGnify beta={m_beta:.4f}")
            print(f"Directional consistency: {'YES' if same_sign else 'NO'}")
            print("H8 (Spearman rho): UNTESTABLE — fewer than 2 shared predictors.")
            print("Reporting directional consistency instead.")
        else:
            print("No shared predictors between SPIRE and MGnify PGLS results.")
            print("H8: UNTESTABLE")
    except Exception as e:
        print(f"Error loading PGLS results: {e}")
else:
    if not spire_pgls.exists():
        print(f"SPIRE PGLS results not found: {spire_pgls}")
    if not mgnify_pgls.exists():
        print(f"MGnify PGLS results not found: {mgnify_pgls}")
    print("H8 evaluation postponed until both PGLS analyses are complete.")



=== Hypothesis H8: Cross-dataset PGLS coefficient correlation ===
Shared predictors: ['ko_per_mb_primary_z']
Only 1 shared predictor (ko_per_mb_primary_z): SPIRE beta=-0.0107, MGnify beta=-0.0471
Directional consistency: YES
H8 (Spearman rho): UNTESTABLE — fewer than 2 shared predictors.
Reporting directional consistency instead.


## Summary

**MGnify extension status:**

- NB01b: Feature matrix construction — pending execution
- NB02b: Mobility prediction (H5/H6/H7) — pending execution
- NB03b: Geographic holdout — pending execution
- NB04b: PGLS validation — pending execution
- NB05: This comparison — results pending from NB01b–NB04b

**Key comparisons:**
1. CV RMSE across datasets and models
2. Target variable distributions (PF1_Cu — both datasets use same target)
3. Cross-dataset PGLS coefficient correlation (H8)
